Azure offers two PaaS compute services that remove VM management entirely:

- **Azure App Service** — a fully managed platform for hosting web applications, REST APIs, and mobile backends. You bring the code; Azure handles the OS, patching, scaling, and availability.
- **Azure Functions** — serverless, event-driven compute. You write a function; Azure runs it in response to a trigger, scales automatically, and charges only for execution time.

Both sit above VMs in the abstraction stack, and both can run on Windows or Linux.

## App Service Plans

An **App Service Plan** defines the region, number of VM instances, and pricing tier for your apps. Multiple App Service apps can share one plan — they all run on the same underlying VM instances.

### Pricing tiers

| Tier | Category | Key features |
|---|---|---|
| **F1** | Free | 60 min/day compute; shared infrastructure; no custom domain |
| **D1** | Shared | 240 min/day; custom domain; still shared |
| **B1/B2/B3** | Basic | Dedicated VMs; manual scale up to 3 instances; no autoscale |
| **S1/S2/S3** | Standard | Autoscale up to 10 instances; deployment slots (5); custom domains + SSL |
| **P1v3/P2v3/P3v3** | Premium v3 | Autoscale up to 30 instances; VNet integration; more memory/CPU |
| **I1v2/I2v2/I3v2** | Isolated v2 | Dedicated VNet; highest scale; App Service Environment v3 |

### Scale up vs Scale out

| | Scale up | Scale out |
|---|---|---|
| What changes | Move to a higher tier (more CPU/RAM per instance) | Add more instances of the same tier |
| Analogy | Bigger VM size | More VMs |
| Requires | Brief restart | No downtime |
| Best for | Memory/CPU bottleneck | Handling more concurrent traffic |

> **Free and Shared tiers** run on shared infrastructure — your app shares CPU with other customers. All other tiers use dedicated VMs.

## App Service Web Apps

An **App Service Web App** is the most common App Service resource — a managed host for a web application or API.

### Supported runtimes

| Runtime | Notes |
|---|---|
| **.NET** | .NET 6, 8, Framework 4.x (Windows only) |
| **Node.js** | 18, 20, 22 LTS |
| **Python** | 3.9, 3.10, 3.11, 3.12 (Linux only) |
| **Java** | 8, 11, 17, 21 — Tomcat, JBoss, or embedded |
| **PHP** | 8.x (Linux) |
| **Ruby** | Linux only |
| **Custom container** | Any Docker image from ACR, Docker Hub, or any registry |

### Configuration

| Setting | Description |
|---|---|
| **App settings** | Environment variables injected at runtime — override `appsettings.json` in .NET |
| **Connection strings** | Special app settings for DB connection strings — shown masked in the portal |
| **Always On** | Keeps the app loaded when idle (prevents cold starts) — Standard tier and above |
| **HTTP/2** | Enabled by default on newer plans |
| **Managed Identity** | System or user-assigned identity for accessing Key Vault, storage, etc. |
| **Custom domain** | Map your own domain + add a TLS certificate (free managed cert available) |

### Built-in authentication (Easy Auth)

App Service has a built-in authentication layer — enable sign-in with Entra ID, Microsoft, Google, Facebook, GitHub, or any OpenID Connect provider with a few clicks and zero code changes.

## Deployment Options

| Method | Description |
|---|---|
| **GitHub Actions** | CI/CD pipeline triggered on push — Azure creates a starter workflow automatically |
| **Azure DevOps** | Full enterprise CI/CD with pipelines |
| **ZIP deploy** | Push a zip file via CLI or REST API — simplest option for scripts |
| **Local Git** | Push to an Azure-hosted Git remote — `git push azure main` |
| **FTP/FTPS** | Legacy; not recommended |
| **Container registry** | Pull and run a Docker image from ACR or Docker Hub |

### Deployment Slots

**Deployment slots** are live, fully functional staging environments within the same App Service app. Each slot has its own hostname and can run a different version of the code.

```
Production slot:  myapp.azurewebsites.net       ← live traffic
Staging slot:     myapp-staging.azurewebsites.net  ← new version deployed here first
```

**Slot swap** exchanges the staging and production slots — swapping their content, app settings, and connection strings — with **zero downtime**. Traffic instantly shifts to the new version.

If the new version has issues, swap back in seconds.

| Tier | Slots available |
|---|---|
| Standard | 5 |
| Premium v3 | 20 |
| Isolated v2 | 20 |

### Sticky settings

By default, app settings are swapped along with the code. Mark a setting as a **slot setting** to make it sticky — it stays with the slot and does not get swapped. Use this for settings that differ between staging and production (e.g. database connection strings, API keys).

## App Service Autoscaling

App Service supports the same Azure Monitor autoscaling as VMSS:

- **Metric-based** — scale out/in based on CPU, memory, HTTP queue length, or custom metrics
- **Scheduled** — scale at specific times for known traffic patterns

Requires **Standard tier or above**.

```
Scale-out:  HTTP Queue Length > 100 for 5 min  →  add 2 instances
Scale-in:   CPU < 20% for 15 min              →  remove 1 instance
```

### App Service Environment (ASE)

An **ASE** is a fully isolated, dedicated App Service deployment inside your own VNet. It provides:
- Complete network isolation — no shared infrastructure
- Direct VNet connectivity to internal resources
- Compliance for regulated industries (PCI, HIPAA)
- Scale up to hundreds of instances

ASEv3 (current version) runs on the Isolated v2 pricing tier and is significantly cheaper and faster to provision than earlier versions.

## Azure Functions

**Azure Functions** is Azure's serverless compute service — the equivalent of AWS Lambda. You write a function, define a trigger, and Azure handles everything else: infrastructure, scaling, patching, and availability.

| | Azure VMs | App Service | Azure Functions |
|---|---|---|---|
| **You manage** | OS, app, scaling | App only | Business logic only |
| **Scaling** | Manual/VMSS | Autoscale | Automatic (0 → N) |
| **Billing** | Per VM-hour | Per plan instance-hour | Per execution + duration |
| **Idle cost** | Yes | Yes (Basic+) | No (Consumption plan) |
| **Max duration** | Unlimited | Unlimited | 5–60 min (plan dependent) |

### Supported languages

.NET (C#/F#), Node.js, Python, Java, PowerShell, and custom handlers (any language that can start an HTTP server).

## Triggers and Bindings

A **trigger** is what starts a function — every function has exactly one. **Bindings** are declarative connections to other services — input bindings read data, output bindings write data.

```
Trigger (1)     →  Function  →  Output Binding (0..n)
Input Binding (0..n)  ↗
```

### Common triggers

| Trigger | When it fires | AWS equivalent |
|---|---|---|
| **HTTP** | Inbound HTTP/HTTPS request | API Gateway → Lambda |
| **Timer** | CRON schedule | CloudWatch Events → Lambda |
| **Blob Storage** | File created/updated in a container | S3 event → Lambda |
| **Queue Storage** | Message added to a queue | SQS → Lambda |
| **Service Bus** | Message/session on a queue or topic | SQS/SNS → Lambda |
| **Event Hub** | Event stream batch | Kinesis → Lambda |
| **Cosmos DB** | Change feed from a Cosmos DB container | DynamoDB Streams → Lambda |
| **Event Grid** | Event published to a topic or subscription | EventBridge → Lambda |
| **SignalR** | Client connection/message | — |

### Example — HTTP trigger (Python)

```python
import azure.functions as func
import json

app = func.FunctionApp()

@app.route(route="hello", methods=["GET", "POST"])
def hello(req: func.HttpRequest) -> func.HttpResponse:
    name = req.params.get('name') or req.get_json().get('name', 'World')
    return func.HttpResponse(json.dumps({'message': f'Hello, {name}!'}),
                              mimetype="application/json")
```

### Example — Timer trigger + Blob output binding

```python
@app.timer_trigger(schedule="0 0 * * * *")   # every hour
@app.blob_output(arg_name="outputBlob",
                 path="reports/{DateTime}.json",
                 connection="AzureWebJobsStorage")
def generate_report(myTimer: func.TimerRequest, outputBlob: func.Out[str]):
    report = {"timestamp": str(myTimer.past_due), "status": "ok"}
    outputBlob.set(json.dumps(report))
```

## Functions Hosting Plans

The hosting plan determines scaling behaviour, cost, and available features.

| Plan | Scaling | Cold start | VNet | Max timeout | Billing |
|---|---|---|---|---|---|
| **Consumption** | 0 → N automatically | Yes | No (HTTP only) | 5 min (default) / 10 min (max) | Per execution + GB-s |
| **Flex Consumption** | 0 → N; configurable concurrency | Reduced | Yes | 60 min | Per execution + GB-s |
| **Premium** | Pre-warmed instances; fast scale | None (pre-warmed) | Yes | Unlimited | Per vCPU-s (always-on) |
| **Dedicated (App Service)** | Manual or autoscale | None | Yes | Unlimited | Per App Service Plan |
| **Container Apps** | 0 → N | Yes | Yes | Unlimited | Per vCPU-s / requests |

### Consumption plan

True serverless — scales to zero, pay only for what you use. Best for event-driven, intermittent workloads. Cold starts (function environment initialisation) can add 100–500ms latency on the first request after idle.

### Premium plan

Pre-warmed instances eliminate cold starts for latency-sensitive functions. Always-on compute means you pay even with zero traffic. Adds VNet integration, longer timeouts, and more powerful instances.

### Flex Consumption (new)

The best of both worlds — scales to zero and charges per execution like Consumption, but adds VNet integration, per-function concurrency controls, and faster scale-out. The recommended plan for new serverless workloads.

## Durable Functions

**Durable Functions** is an extension that lets you write **stateful, long-running workflows** in Azure Functions using ordinary code — no external state management required. Azure handles checkpointing, retrying, and resuming automatically.

This is Azure's equivalent of AWS Step Functions, but expressed as code rather than JSON state machine definitions.

### Function types

| Type | Role |
|---|---|
| **Orchestrator** | Defines the workflow — calls activity functions and coordinates the flow |
| **Activity** | The actual units of work — called by the orchestrator |
| **Client (Starter)** | Starts a new orchestration instance — can be triggered by HTTP, queue, etc. |
| **Entity** | Manages a small piece of durable state (like an actor) |

### Common patterns

**Function chaining** — execute activities sequentially, passing output of one to the next:

```python
@app.orchestration_trigger(context_name="context")
def my_orchestrator(context: df.DurableOrchestrationContext):
    result1 = yield context.call_activity("Step1", "input")
    result2 = yield context.call_activity("Step2", result1)
    result3 = yield context.call_activity("Step3", result2)
    return result3
```

**Fan-out / Fan-in** — run many activities in parallel, then aggregate results:

```python
@app.orchestration_trigger(context_name="context")
def parallel_orchestrator(context: df.DurableOrchestrationContext):
    items = ["item1", "item2", "item3", "item4"]
    tasks = [context.call_activity("ProcessItem", item) for item in items]
    results = yield context.task_all(tasks)   # wait for ALL to complete
    return sum(results)
```

**Other patterns:**
- **Async HTTP API** — start a long operation, poll for completion via status endpoint
- **Monitoring** — poll an external condition until it is met
- **Human interaction** — pause the workflow and wait for an external approval signal

## Choosing Between App Service and Functions

| Scenario | Use |
|---|---|
| Full web application or REST API with persistent state | App Service Web App |
| Containerised web app | App Service (custom container) or AKS |
| Event-driven, short-lived code (file upload, queue consumer) | Azure Functions (Consumption) |
| Low-latency API with no cold start tolerance | Azure Functions (Premium / Flex) |
| Scheduled background jobs | Azure Functions (Timer trigger) |
| Complex multi-step workflows | Durable Functions |
| App needing VNet integration on a tight budget | App Service or Functions Flex |
| Compliance-driven isolation (PCI, HIPAA) | App Service Environment (ASE) |

## Working with App Service and Functions via Python

In [ ]:
# pip install azure-identity azure-mgmt-web

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.web import WebSiteManagementClient

credential = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"

web = WebSiteManagementClient(credential, subscription_id)

In [ ]:
# List all App Service apps and Function Apps in the subscription
for app in web.web_apps.list():
    kind = app.kind or "app"
    state = app.state
    plan = app.server_farm_id.split("/")[-1] if app.server_farm_id else "-"
    print(f"{app.name:<40} {kind:<20} {state:<10} plan: {plan}")

In [ ]:
# List deployment slots for a specific app
app_name = "my-web-app"
slots = web.web_apps.list_slots(resource_group, app_name)
for slot in slots:
    print(f"  Slot: {slot.name:<30} State: {slot.state}  URL: https://{slot.default_host_name}")

In [ ]:
from azure.mgmt.web.models import CsmSlotEntity

# Swap staging slot to production (zero-downtime deploy)
poller = web.web_apps.begin_swap_slot(
    resource_group,
    app_name,
    slot="staging",
    slot_swap_entity=CsmSlotEntity(target_slot="production", preserve_vnet=True)
)
poller.result()
print(f"Swap complete — staging is now live on {app_name}.azurewebsites.net")

In [ ]:
from azure.mgmt.web.models import StringDictionary

# Read current app settings
settings = web.web_apps.list_application_settings(resource_group, app_name)
for key, val in settings.properties.items():
    print(f"  {key} = {val}")

# Update app settings (merges with existing)
web.web_apps.update_application_settings(
    resource_group,
    app_name,
    app_settings=StringDictionary(properties={"ENVIRONMENT": "production", "LOG_LEVEL": "warn"})
)
print("Settings updated")

## Summary

| Concept | Key Takeaway |
|---|---|
| App Service Plan | Defines region, VM tier, and instance count — multiple apps share one plan |
| Scale up | Move to a bigger tier (more CPU/RAM per instance) |
| Scale out | Add more instances of the same tier — requires Standard tier or above |
| Deployment slots | Live staging environments; swap to production with zero downtime |
| Sticky settings | App settings that stay with the slot and are not swapped |
| Easy Auth | Built-in auth with Entra ID / social providers — no code changes needed |
| App Service Environment | Fully isolated, VNet-injected App Service for compliance workloads |
| Azure Functions | Serverless, event-driven; one trigger per function; bindings for I/O |
| Consumption plan | True serverless — scales to zero, pay per execution; cold starts possible |
| Flex Consumption | Scales to zero + VNet integration + per-function concurrency — recommended for new functions |
| Premium plan | Pre-warmed — no cold starts; always-on billing; VNet integration |
| Durable Functions | Stateful workflows in code — chaining, fan-out/fan-in, async HTTP, human approval |
| App Service vs Functions | App Service for persistent web apps; Functions for event-driven, short-lived work |